# Within-Domain Evaluation
## Description
In order to assess the models' performance, we use evaluation metrics that analyze their ability to predict user preferences within-domain. This is accomplished with two calculation methods. First, we apply Spark's RankingMetrics to compare top-15 recommendations with actual user preferences in held-out test data. Then, the second approach calculates manual per-user metrics, with a different number of users with both predictions and test positives. 

Each method is applied in the same manner to both book and movie datasets. The ones implemented in the analysis were test datasets, which is the output of a prior split into training and testing sets, and user recommendation datasets, consisting of users' id's and hybrid recommendations = item id + recommendation.

## Set-Up

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("RecommenderEvaluation").getOrCreate()

In [2]:
from pyspark.sql.functions import (
    col, explode, collect_list, from_json, 
    array_contains, when, lit, row_number, 
    sum, count, avg, size
)
from pyspark.sql.window import Window
from pyspark.sql.types import ArrayType, DoubleType
from pyspark.mllib.evaluation import RankingMetrics
import logging

# Set up basic logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Books

## 0. Loading Data

In [3]:
try:
    # Load recommendations
    logger.info("Loading recommendation data...")
    book_recs = spark.read.csv("./Models/models_results/book_user_recs.csv", header=True, inferSchema=True)
    book_recs.show(3)

    # Load test data
    logger.info("Loading test data...")
    book_test = spark.read.csv("./Models/test_data/test_book_data.csv", header=True, inferSchema=True)

except Exception as e:
    logger.error(f"Error loading data files: {str(e)}")
    raise

# Parse recommendations
recommendation_schema = ArrayType(ArrayType(DoubleType()))
book_recs_parsed = book_recs.withColumn(
    "hybrid_recommendations",
    from_json(col("hybrid_recommendations_json"), recommendation_schema)
).drop("hybrid_recommendations_json")
book_recs_parsed = book_recs_parsed.filter(col("hybrid_recommendations").isNotNull())

# Explode recommendations
book_recs_exploded = book_recs_parsed.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).cast("long").alias("item_id"),
    col("rec").getItem(1).alias("score")
)

INFO:__main__:Loading recommendation data...
INFO:__main__:Loading test data...


+-------+---------------------------+
|user_id|hybrid_recommendations_json|
+-------+---------------------------+
|    148|       [[2963218.0,0.659...|
|    463|       [[1.3341979E7,0.8...|
|    471|       [[4.7815497E7,0.6...|
+-------+---------------------------+
only showing top 3 rows



## 1. Ranking Metrics

We assign a K-value and a rating threshold to identify items that were liked by users.

In [4]:
# Debug info
print("=== DEBUG INFO ===")
print(f"Test data size: {book_test.count()}")
print(f"Unique users in test: {book_test.select('user_id').distinct().count()}")
print(f"Recommendations size: {book_recs_exploded.count()}")
print(f"Unique users in recs: {book_recs_exploded.select('user_id').distinct().count()}")

# Ranking evaluation parameters
K = 15
rating_threshold = 4.0

=== DEBUG INFO ===
Test data size: 1030660
Unique users in test: 209211
Recommendations size: 4958460
Unique users in recs: 330564


In [5]:
# 1) First get all test items for each user (both positive and negative)
test_items_by_user = book_test.groupBy("user_id").agg(
    collect_list("item_id").alias("test_items")
)

By joining the recommendations with the test dataset, we can identify known recommendations and filter by them.

In [6]:
# 2) Join recommendations with test items 
known_recommendations = book_recs_exploded.join(
    test_items_by_user,
    on="user_id",
    how="inner"
).withColumn(
    "is_known",
    array_contains(col("test_items"), col("item_id"))
).filter(col("is_known"))

In [7]:
# 3) Get top-K known recommendations per user
w = Window.partitionBy("user_id").orderBy(col("score").desc())
preds_known = known_recommendations \
    .withColumn("rank", row_number().over(w)) \
    .filter(col("rank") <= K) \
    .groupBy("user_id") \
    .agg(collect_list("item_id").alias("predicted"))

In [8]:
# 4) Get true positives (items rated >= threshold)
test_pos = book_test \
    .filter(col("rating") >= rating_threshold) \
    .groupBy("user_id") \
    .agg(collect_list("item_id").alias("true_positives"))

In [9]:
# 5) Join only users who have both predictions and positives
preds_and_truth_known = preds_known.join(test_pos, on="user_id", how="inner") \
    .rdd \
    .map(lambda r: (r.predicted, r.true_positives))

print(f"\nUsers with both predictions and test positives: {preds_and_truth_known.count()}")


Users with both predictions and test positives: 9542


In the next step, we work with known items only, since using empty recommendations will hinder the analysis process.

In [10]:
# 6) Compute standard ranking metrics 
if preds_and_truth_known.count() > 0:
    try:
        metrics = RankingMetrics(preds_and_truth_known)
        print(f"\nRanking Metrics @{K} (KNOWN ITEMS ONLY):")
        precision_k = metrics.precisionAt(K)
        recall_k = metrics.recallAt(K)
        f1_k = 2 * (precision_k * recall_k) / (precision_k + recall_k) if (precision_k + recall_k) > 0 else 0
        
        print(f"Precision@{K}: {precision_k:.4f}")
        print(f"Recall@{K}   : {recall_k:.4f}")
        print(f"F1@{K}       : {f1_k:.4f}")
        print(f"NDCG@{K}     : {metrics.ndcgAt(K):.4f}")
        
        # Additional metrics for different K values
        print(f"\nDetailed metrics for different K values:")
        for k in [5, 10, 15, 20]:
            if k <= K:
                prec_k = metrics.precisionAt(k)
                rec_k = metrics.recallAt(k)
                f1_k = 2 * (prec_k * rec_k) / (prec_k + rec_k) if (prec_k + rec_k) > 0 else 0
                print(f"K={k:2d} - Precision: {prec_k:.4f}, Recall: {rec_k:.4f}, F1: {f1_k:.4f}")
    except Exception as e:
        logger.error(f"Error computing ranking metrics: {str(e)}")


Ranking Metrics @15 (KNOWN ITEMS ONLY):
Precision@15: 0.0711
Recall@15   : 0.3586
F1@15       : 0.1187
NDCG@15     : 0.4593

Detailed metrics for different K values:
K= 5 - Precision: 0.2134, Recall: 0.3586, F1: 0.2676
K=10 - Precision: 0.1067, Recall: 0.3586, F1: 0.1645
K=15 - Precision: 0.0711, Recall: 0.3586, F1: 0.1187


The output reveals that only 7.1% of recommended books were actually relevant (precision), while capturing 35.9% of liked books (recall). The normalized discounted cumulative gain (NDCG) score of 0.459 shows a relative success in ranking relevant books higher. 

Although this precision score is not desirable, the reason it is this low lies in the method's algorithm fixing K as a denominator for all users, while only taking users that have both predictions and test positives. Moreover, Ranking Metrics gives equal weight to all users' top-K lists, which contributed to such an output.

These metrics show the system retrieves a reasonable portion of preferred books but struggles with recommendation precision, suggesting some irrelevant suggestions. 

## 2. Per-User Metrics

This code calculates precision and recall for each user individually, then averages the results. 

In [11]:
print(f"\n=== DETAILED PER-USER METRICS ===")

# Get top-K known recommendations as DataFrame
preds_known_df = known_recommendations \
    .withColumn("rank", row_number().over(w)) \
    .filter(col("rank") <= K)

# Join with test data to get actual ratings
preds_with_ratings = preds_known_df.join(
    book_test.select("user_id", "item_id", "rating"),
    on=["user_id", "item_id"],
    how="left"
)

# Calculate true positives count per user
tp_counts = preds_with_ratings.filter(col("rating") >= rating_threshold) \
    .groupBy("user_id") \
    .agg(count("*").alias("tp"))

# Calculate false positives count per user
fp_counts = preds_with_ratings.filter(
    (col("rating") < rating_threshold) & 
    (col("rating").isNotNull())
).groupBy("user_id") \
 .agg(count("*").alias("fp"))

# Get total recommendations per user
total_recs = preds_with_ratings.groupBy("user_id") \
    .agg(count("*").alias("total_recs"))

# Join all metrics together
user_metrics = total_recs.join(tp_counts, "user_id", "left") \
    .join(fp_counts, "user_id", "left") \
    .fillna(0, ["tp", "fp"])

# Join with total positives count from test data
user_metrics = user_metrics.join(
    test_pos.select("user_id", size("true_positives").alias("total_positives")),
    on="user_id",
    how="left"
).fillna(0, subset=["total_positives"])

# Compute per-user precision and recall
user_metrics = user_metrics.withColumn(
    "user_precision", col("tp") / col("total_recs")
).withColumn(
    "user_recall",
    when(col("total_positives") > 0, col("tp") / col("total_positives")).otherwise(0)
)

# Aggregate results
global_avg_metrics = user_metrics.agg(
    avg("user_precision").alias("avg_precision"),
    avg("user_recall").alias("avg_recall"),
    count("*").alias("n_users")
).collect()[0]

print(f"\nAverage per-user metrics (across {global_avg_metrics['n_users']} users):")
print(f"  Precision@{K}: {global_avg_metrics['avg_precision']:.4f}")
print(f"  Recall@{K}:    {global_avg_metrics['avg_recall']:.4f}")


=== DETAILED PER-USER METRICS ===

Average per-user metrics (across 9713 users):
  Precision@15: 0.9425
  Recall@15:    0.3522


This implementation yields higher precision scores (94,25%) than Spark's RankingMetrics, because it uses a recommendation lists. Instead of using K in the denominator, it divides true positives by each user's actual number of recommendations (which may be fewer than K). 

This makes precision seem better than it would be in a stricter evaluation with a full set of recommendations.

# Movies

The same logic of operations is applied to the movie datasets.

## 0. Loading Data

In [12]:
try:
    # Load recommendations
    logger.info("Loading recommendation data...")
    movie_recs = spark.read.csv("./Models/models_results/movie_user_recs.csv", header=True, inferSchema=True)
    movie_recs.show(3)

    # Load test data
    logger.info("Loading test data...")
    movie_test = spark.read.csv("./Models/test_data/test_movie_data.csv", header=True, inferSchema=True)

except Exception as e:
    logger.error(f"Error loading data files: {str(e)}")
    raise

# Parse recommendations
recommendation_schema = ArrayType(ArrayType(DoubleType()))
movie_recs_parsed = movie_recs.withColumn(
    "hybrid_recommendations",
    from_json(col("hybrid_recommendations_json"), recommendation_schema)
).drop("hybrid_recommendations_json")
movie_recs_parsed = movie_recs_parsed.filter(col("hybrid_recommendations").isNotNull())

# Explode recommendations
movie_recs_exploded = movie_recs_parsed.withColumn(
    "rec", explode("hybrid_recommendations")
).select(
    "user_id",
    col("rec").getItem(0).cast("long").alias("item_id"),
    col("rec").getItem(1).alias("score")
)

INFO:__main__:Loading recommendation data...
INFO:__main__:Loading test data...


+-------+---------------------------+
|user_id|hybrid_recommendations_json|
+-------+---------------------------+
|   7253|       [[86892.0,0.66435...|
|  26755|       [[78499.0,0.74189...|
|  28088|       [[78499.0,0.65407...|
+-------+---------------------------+
only showing top 3 rows



## 1. Ranking Metrics

In [14]:
# Debug info
print("=== DEBUG INFO ===")
print(f"Test data size: {movie_test.count()}")
print(f"Unique users in test: {movie_test.select('user_id').distinct().count()}")
print(f"Recommendations size: {movie_recs_exploded.count()}")
print(f"Unique users in recs: {movie_recs_exploded.select('user_id').distinct().count()}")

# Ranking evaluation parameters
K = 15
rating_threshold = 4.0

=== DEBUG INFO ===
Test data size: 227110
Unique users in test: 21195
Recommendations size: 322440
Unique users in recs: 21496


In [15]:
# 1) First get all test items for each user (both positive and negative)
test_items_by_user = movie_test.groupBy("user_id").agg(
    collect_list("item_id").alias("test_items")
)

In [17]:
# 2) Join recommendations with test items to identify known recommendations
known_recommendations = movie_recs_exploded.join(
    test_items_by_user,
    on="user_id",
    how="inner"
).withColumn(
    "is_known",
    array_contains(col("test_items"), col("item_id"))
).filter(col("is_known"))

In [18]:
# 3) Get top-K known recommendations per user
w = Window.partitionBy("user_id").orderBy(col("score").desc())
preds_known = known_recommendations \
    .withColumn("rank", row_number().over(w)) \
    .filter(col("rank") <= K) \
    .groupBy("user_id") \
    .agg(collect_list("item_id").alias("predicted"))

In [19]:
# 4) Get true positives (items rated >= threshold)
test_pos = movie_test \
    .filter(col("rating") >= rating_threshold) \
    .groupBy("user_id") \
    .agg(collect_list("item_id").alias("true_positives"))

In [20]:
# 5) Join only users who have both predictions and positives
preds_and_truth_known = preds_known.join(test_pos, on="user_id", how="inner") \
    .rdd \
    .map(lambda r: (r.predicted, r.true_positives))

print(f"\nUsers with both predictions and test positives: {preds_and_truth_known.count()}")


Users with both predictions and test positives: 16


In [21]:
# 6) Compute standard ranking metrics (KNOWN ITEMS ONLY)
if preds_and_truth_known.count() > 0:
    try:
        metrics = RankingMetrics(preds_and_truth_known)
        print(f"\nRanking Metrics @{K} (KNOWN ITEMS ONLY):")
        precision_k = metrics.precisionAt(K)
        recall_k = metrics.recallAt(K)
        f1_k = 2 * (precision_k * recall_k) / (precision_k + recall_k) if (precision_k + recall_k) > 0 else 0
        
        print(f"Precision@{K}: {precision_k:.4f}")
        print(f"Recall@{K}   : {recall_k:.4f}")
        print(f"F1@{K}       : {f1_k:.4f}")
        print(f"NDCG@{K}     : {metrics.ndcgAt(K):.4f}")
        
        # Additional metrics for different K values
        print(f"\nDetailed metrics for different K values:")
        for k in [5, 10, 15, 20]:
            if k <= K:
                prec_k = metrics.precisionAt(k)
                rec_k = metrics.recallAt(k)
                f1_k = 2 * (prec_k * rec_k) / (prec_k + rec_k) if (prec_k + rec_k) > 0 else 0
                print(f"K={k:2d} - Precision: {prec_k:.4f}, Recall: {rec_k:.4f}, F1: {f1_k:.4f}")
    except Exception as e:
        logger.error(f"Error computing ranking metrics: {str(e)}")


Ranking Metrics @15 (KNOWN ITEMS ONLY):
Precision@15: 0.0667
Recall@15   : 0.2434
F1@15       : 0.1047
NDCG@15     : 0.3619

Detailed metrics for different K values:
K= 5 - Precision: 0.2000, Recall: 0.2434, F1: 0.2196
K=10 - Precision: 0.1000, Recall: 0.2434, F1: 0.1418
K=15 - Precision: 0.0667, Recall: 0.2434, F1: 0.1047


These results show the recommender system's performance when suggesting up to 15 movies. 

It achieves modest accuracy (6.7% precision at K=15), meaning only approximately 1 in 15 recommended movies matches what users really liked. Recall (24.3%) indicates that the system finds about 1/4 of preferred movies, while the low F1-score (10.5%) and NDCG (0.36) show room for improvement in ranking quality. 

Performance improves with smaller K values, as the system's top recommendations tend to be more relevant. The same recall across K values implies the system spreads correct predictions throughout the recommendation list rather than concentrates them.  

## 2. Per-User Metrics

In [23]:
print(f"\n=== DETAILED PER-USER METRICS ===")

# Get top-K known recommendations as DataFrame
preds_known_df = known_recommendations \
    .withColumn("rank", row_number().over(w)) \
    .filter(col("rank") <= K)

# Join with test data to get actual ratings
preds_with_ratings = preds_known_df.join(
    movie_test.select("user_id", "item_id", "rating"),
    on=["user_id", "item_id"],
    how="left"
)

# Calculate true positives count per user
tp_counts = preds_with_ratings.filter(col("rating") >= rating_threshold) \
    .groupBy("user_id") \
    .agg(count("*").alias("tp"))

# Calculate false positives count per user
fp_counts = preds_with_ratings.filter(
    (col("rating") < rating_threshold) & 
    (col("rating").isNotNull())
).groupBy("user_id") \
 .agg(count("*").alias("fp"))

# Get total recommendations per user
total_recs = preds_with_ratings.groupBy("user_id") \
    .agg(count("*").alias("total_recs"))

# Join all metrics together
user_metrics = total_recs.join(tp_counts, "user_id", "left") \
    .join(fp_counts, "user_id", "left") \
    .fillna(0, ["tp", "fp"])

# Join with total positives count from test data
user_metrics = user_metrics.join(
    test_pos.select("user_id", size("true_positives").alias("total_positives")),
    on="user_id",
    how="left"
).fillna(0, subset=["total_positives"])

# Compute per-user precision and recall
user_metrics = user_metrics.withColumn(
    "user_precision", col("tp") / col("total_recs")
).withColumn(
    "user_recall",
    when(col("total_positives") > 0, col("tp") / col("total_positives")).otherwise(0)
)

# Aggregate results
global_avg_metrics = user_metrics.agg(
    avg("user_precision").alias("avg_precision"),
    avg("user_recall").alias("avg_recall"),
    count("*").alias("n_users")
).collect()[0]

print(f"\nAverage per-user metrics (across {global_avg_metrics['n_users']} users):")
print(f"  Precision@{K}: {global_avg_metrics['avg_precision']:.4f}")
print(f"  Recall@{K}:    {global_avg_metrics['avg_recall']:.4f}")


=== DETAILED PER-USER METRICS ===

Average per-user metrics (across 16 users):
  Precision@15: 0.8750
  Recall@15:    0.2434


The per-user evaluation shows much higher precision (87.5%) than the RankingMetrics approach, which is similar to what we've seen with the books dataset. Recall, however, remains identical (24.3%). 

The increased precision occurs because the system calculates metrics for only 16 users with complete data, rather than the full test set. This means that the method may overestimate performance, since it focuses on users where the recommender worked correctly. 

Still, the consistent recall shows that the system finds only about a quarter of liked movies, regardless of evaluation method.